A prompt is the input or instruction given to an AI model to tell it what task to perform and what kind of response is expected.

A prompt can contain:

Instructions
Questions
Context
Examples
Constraints
Output format

Instruction:
Summarize the following customer feedback.

Question:
What are the main issues reported by customers?

Context:
The feedback is collected from users of an online food delivery app.

Examples:
Input: "Delivery was late but food quality was good."
Output: "Delivery Issue"

Constraints:
- Use only the provided feedback
- Do not add assumptions
- Keep the answer under 100 words

Output Format:
Return the result as:
1. Issue
2. Frequency
3. Short explanation

Zero-Shot Prompting

The model is given a task without any examples.

Example:

Classify the sentiment:
"The service was excellent."
One-Shot Prompting

The model is given one example before the actual task.

Example:

Example:
"The product is amazing." → Positive

Now classify:
"The service was terrible."
Few-Shot Prompting

The model is given multiple examples before the actual task.

Example:

"The product is amazing." → Positive
"The service was terrible." → Negative
"The food was okay." → Neutral

Now classify:
"The delivery was very fast."

Simple difference:

Zero-shot → 0 examples
One-shot  → 1 example
Few-shot  → Multiple examples

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template(
    """
You are an AI instructor.

Explain {topic} to {audience}.

Requirements:
- Use simple English
- Include one practical example
- Keep the answer under {word_limit} words
"""
)

model = ChatOpenAI(
    model="gpt-5.6"
)

chain = prompt | model

response = chain.invoke(
    {
        "topic": "Vector Database",
        "audience": "beginner developers",
        "word_limit": 200
    }
)

print(response.content)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an experienced AI instructor. Use simple English."
        ),
        (
            "human",
            "Explain {topic} to {audience}. Include one example."
        )
    ]
)

model = ChatOpenAI(model="gpt-5.6")

chain = prompt | model

response = chain.invoke(
    {
        "topic": "Vector Database",
        "audience": "beginner developers"
    }
)

print(response.content)

PromptTemplate
→ Creates a single text prompt

ChatPromptTemplate
→ Creates structured chat messages with roles
   such as system, human, and assistant

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
You are an AI instructor.

Explain {{ topic }} to {{ audience }}.

{% if include_example %}
Include one practical example.
{% endif %}

{% if level == "beginner" %}
Use very simple English and avoid complex terminology.
{% elif level == "advanced" %}
Include technical details and architecture.
{% else %}
Use moderate technical depth.
{% endif %}
"""

prompt = PromptTemplate.from_template(
    template,
    template_format="jinja2"
)

formatted_prompt = prompt.invoke(
    {
        "topic": "RAG",
        "audience": "Python developers",
        "include_example": True,
        "level": "beginner"
    }
)

print(formatted_prompt.to_string())

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
You are a customer support assistant.

Customer Query:
{{ query }}

{% if user_type == "premium" %}
Provide a detailed response and mention priority support.
{% else %}
Provide a concise response.
{% endif %}
"""

prompt = PromptTemplate.from_template(
    template,
    template_format="jinja2"
)

result = prompt.invoke(
    {
        "query": "My payment failed.",
        "user_type": "premium"
    }
)

print(result.to_string())

Jinja prompting allows us to dynamically change parts of a prompt using variables, conditions, and loops.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI instructor.

{% if level == "beginner" %}
Use very simple English and avoid complex terminology.
{% elif level == "advanced" %}
Include technical details and architecture.
{% else %}
Use moderate technical depth.
{% endif %}
"""
        ),
        (
            "human",
            """
Explain {{ topic }} to {{ audience }}.

{% if include_example %}
Include one practical example.
{% endif %}
"""
        )
    ],
    template_format="jinja2"
)

formatted_prompt = prompt.invoke(
    {
        "topic": "RAG",
        "audience": "Python developers",
        "include_example": True,
        "level": "beginner"
    }
)

for message in formatted_prompt.to_messages():
    print(message.type.upper())
    print(message.content)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an enterprise AI assistant.

Follow these rules:

{% for rule in rules %}
- {{ rule }}
{% endfor %}
"""
        ),
        (
            "human",
            """
Question:
{{ question }}

{% if output_format == "json" %}
Return the response in JSON format.
{% else %}
Return the response in Markdown.
{% endif %}
"""
        )
    ],
    template_format="jinja2"
)

result = prompt.invoke(
    {
        "rules": [
            "Do not fabricate information",
            "Keep the answer concise",
            "Use only the supplied context"
        ],
        "question": "What is RAG?",
        "output_format": "json"
    }
)

for message in result.to_messages():
    print(message.type, ":", message.content)

PromptTemplate + Jinja
→ Dynamic single-text prompt

ChatPromptTemplate + Jinja
→ Dynamic role-based chat prompt
   + variables
   + if/else
   + loops

In [ ]:
import json

with open("prompts.json", "r", encoding="utf-8") as file:
    prompts = json.load(file)

prompt = prompts["rag_prompt"]

final_prompt = prompt.format(
    context="Employees receive 24 paid leaves every year.",
    question="How many paid leaves do employees receive?"
)

print(final_prompt)

In [ ]:
import json

from langchain_core.prompts import ChatPromptTemplate


def load_prompt(prompt_name):
    
    with open("prompt.json", "r", encoding="utf-8") as file:
        prompts = json.load(file)

    if prompt_name not in prompts:
        raise ValueError(
            f"Prompt '{prompt_name}' not found."
        )

    config = prompts[prompt_name]

    messages = [
        (message["role"], message["template"])
        for message in config["messages"]
    ]

    prompt = ChatPromptTemplate.from_messages(
        messages,
        template_format=config["template_format"]
    )

    return prompt

In [ ]:
prompt = load_prompt("rag_prompt")

result = prompt.invoke(
    {
        "context": """
        Employees receive 24 paid leaves every year.
        Maximum 10 unused leaves can be carried forward.
        """,

        "question": "How many paid leaves are available?"
    }
)

for message in result.to_messages():
    print(message.type.upper())
    print(message.content)

In [ ]:
prompt = load_prompt("rag_prompt")

result = prompt.invoke(
    {
        "context": """
        Employees receive 24 paid leaves every year.
        Maximum 10 unused leaves can be carried forward.
        """,

        "question": "How many paid leaves are available?"
    }
)

for message in result.to_messages():
    print(message.type.upper())
    print(message.content)

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5.6"
)

prompt = load_prompt(
    "rag_prompt"
)

chain = prompt | model

response = chain.invoke(
    {
        "context": """
        Employees receive 24 paid leaves annually.
        """,

        "question":
        "How many paid leaves do employees receive?"
    }
)

print(response.content)

prompts.json
     │
     ├── rag_prompt
     ├── summarization_prompt
     ├── classification_prompt
     └── code_review_prompt
              ↓
        load_prompt()
              ↓
     ChatPromptTemplate
              ↓
             LLM

In [ ]:
load_prompt("rag_prompt")

load_prompt("summarization_prompt")

load_prompt("classification_prompt")

load_prompt("code_review_prompt")

In [ ]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate

client = Client()

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple English."
)

client.push_prompt(
    "ai-teaching-prompt",
    object=prompt
)

In [ ]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "ai-teaching-prompt"
)

result = prompt.invoke({
    "topic": "RAG"
})

print(result)